# TP XGBoost — Les hyperparamètres et leur effet sur l'apprentissage

On prédit si un vol arrive avec **plus de 15 minutes de retard** (`nycflights13`). L'objectif de ce TP n'est pas seulement de faire tourner un modèle, mais de **voir comment les principaux hyperparamètres de XGBoost influencent l'apprentissage**.

À chaque étape : une **question est posée à la salle**, on complète le trou `____`, on lance, et on observe l'effet sur l'**AUC d'entraînement** et l'**AUC de test**. Les cellules 🟦 sont des questions d'interprétation à discuter ensemble.

In [ ]:
library(dplyr)
library(nycflights13)
library(xgboost)
library(pROC)

## 1. Préparation des données

On prépare les données une fois pour toutes (variable cible binaire, split train/test 70/30). Rien à compléter ici : on exécute et on passe à la suite.

In [ ]:
data(flights)

set.seed(123)
flights_clean <- flights |>
  filter(!is.na(arr_delay), !is.na(dep_delay), !is.na(air_time)) |>
  mutate(retard = ifelse(arr_delay > 15, 1, 0)) |>
  select(retard, dep_delay, air_time, distance, month, day, hour, minute) |>
  sample_n(50000)

set.seed(123)
idx <- as.vector(caret::createDataPartition(factor(flights_clean$retard), p = 0.70, list = FALSE))
train_flights <- flights_clean[idx, ]
test_flights  <- flights_clean[-idx, ]

X_train <- train_flights |> select(-retard) |> as.matrix()
y_train <- train_flights$retard
X_test  <- test_flights  |> select(-retard) |> as.matrix()
y_test  <- test_flights$retard

cat("Données prêtes :", nrow(X_train), "vols en train,", nrow(X_test), "en test")

## 2. Une fonction pour tester un réglage

Pour comparer facilement les hyperparamètres, on écrit une petite fonction qui entraîne un modèle et affiche **l'AUC sur le train ET sur le test**. Comparer les deux est la clé pour repérer le surapprentissage.

In [ ]:
evaluer <- function(eta = 0.1, max_depth = 6, nrounds = 100) {
  m <- xgboost(data = X_train, label = y_train,
               objective = "binary:logistic",
               eta = eta, max_depth = max_depth, nrounds = nrounds,
               verbose = 0)
  a_train <- auc(roc(y_train, predict(m, X_train), quiet = TRUE))
  a_test  <- auc(roc(y_test,  predict(m, X_test),  quiet = TRUE))
  cat("eta =", eta, "| max_depth =", max_depth, "| nrounds =", nrounds, "\n")
  cat("   AUC train =", round(a_train, 4), " | AUC test =", round(a_test, 4), "\n")
}

## 3. Le taux d'apprentissage (`eta`)

`eta` règle le poids de chaque nouvel arbre dans l'ensemble.

**Question à la salle :** à ton avis, si on prend un `eta` très petit (0.01) avec seulement 100 arbres, le modèle va-t-il mieux ou moins bien apprendre qu'avec `eta = 0.3` ?

*Complète, lance avec `0.01`, puis relance avec `0.3` et compare.*

In [ ]:
evaluer(eta = ____)

> 🟦 **Interprétation.** Comment `eta` influence-t-il l'apprentissage ? (pense à la vitesse d'apprentissage et à ce qui se passe quand il est trop petit avec peu d'arbres)
>
> *Réponds ici :*

## 4. La profondeur des arbres (`max_depth`)

`max_depth` contrôle la complexité de chaque arbre.

**Question à la salle :** si on autorise des arbres très profonds (`max_depth = 12`), le modèle va-t-il forcément mieux généraliser ?

*Complète, lance avec `1` (arbres très simples), puis avec `12` (arbres profonds), et regarde l'écart entre AUC train et AUC test.*

In [ ]:
evaluer(max_depth = ____)

> 🟦 **Interprétation.** Avec `max_depth = 12`, compare l'AUC train et l'AUC test. Que remarques-tu sur l'écart entre les deux ? Comment s'appelle ce phénomène ?
>
> *Réponds ici :*

## 5. Le nombre d'arbres (`nrounds`)

**Question à la salle :** plus on ajoute d'arbres, mieux c'est ? Jusqu'où ?

*Complète, lance avec `10` (peu d'arbres), puis avec `500` (beaucoup), et compare les deux AUC.*

In [ ]:
evaluer(nrounds = ____)

> 🟦 **Interprétation.** En passant de 10 à 500 arbres, comment évoluent l'AUC train et l'AUC test ? Le gain sur le test continue-t-il indéfiniment ?
>
> *Réponds ici :*

## 6. Laisser la validation croisée choisir

Régler les hyperparamètres à la main est fastidieux. La **validation croisée** avec arrêt anticipé choisit automatiquement un bon nombre d'arbres. On exécute et on regarde le résultat.

In [ ]:
X_mat <- xgb.DMatrix(X_train, label = y_train)

set.seed(123)
sel <- xgb.cv(
  data    = X_mat,
  params  = list(objective = "binary:logistic", eta = 0.1, max_depth = 6),
  nrounds = 300, metrics = "auc", nfold = 10,
  early_stopping_rounds = 10, verbose = 0
)

cat("Nombre d'arbres choisi automatiquement :", sel$best_iteration)

> 🟦 **Interprétation.** Pourquoi laisser la validation croisée choisir `nrounds` plutôt que de le fixer soi-même ? Qu'apporte `early_stopping_rounds` ?
>
> *Réponds ici :*

## À retenir

- **`eta`** (taux d'apprentissage) : trop petit → apprentissage lent (sous-apprentissage si peu d'arbres) ; plus grand → apprend plus vite.
- **`max_depth`** : trop grand → arbres complexes qui **surapprennent** (AUC train ↑↑, AUC test qui stagne ou baisse).
- **`nrounds`** : plus d'arbres aide, mais avec un **rendement décroissant** ; au-delà d'un point, le test ne progresse plus.
- La **validation croisée + arrêt anticipé** choisit automatiquement un bon nombre d'arbres, sans tâtonner.